In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models

#### data preprocessing

In [2]:
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Running on:", device)

Running on: cuda


In [3]:
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor()
])

#### load dataset

In [4]:
train = datasets.MNIST(".", train=True, download=True, transform=transform)
loader = torch.utils.data.DataLoader(train, batch_size=64, shuffle=True)

#### load pretrained model

In [5]:
model = models.mobilenet_v2(pretrained=True)

C:\Users\DELL\AppData\Roaming\Python\Python312\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\DELL\AppData\Roaming\Python\Python312\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


#### freeze layers

In [6]:
for param in model.parameters():
    param.requires_grad = False

In [7]:
# unfreeze last conv block
for param in model.features[-1].parameters():
    param.requires_grad = True

# unfreeze classifier
for param in model.classifier.parameters():
    param.requires_grad = True

#### replace classifier

In [8]:
model.classifier[1] = nn.Linear(model.last_channel, 10)

In [9]:
model = model.to(device)

#### optimizer

In [10]:
opt = optim.Adam(model.classifier.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

#### training

In [11]:
for epoch in range(2):
    for i,(x,y) in enumerate(loader):

        x = x.to(device)
        y = y.to(device)

        opt.zero_grad()
        out = model(x)
        loss = loss_fn(out,y)
        loss.backward()
        opt.step()

        if i % 200 == 0:
            print("epoch",epoch,"step",i,"loss",loss.item())

    print("epoch done")

epoch 0 step 0 loss 2.424428939819336
epoch 0 step 200 loss 0.4541901648044586
epoch 0 step 400 loss 0.30602261424064636
epoch 0 step 600 loss 0.3106929659843445
epoch 0 step 800 loss 0.38848546147346497
epoch done
epoch 1 step 0 loss 0.3348187804222107
epoch 1 step 200 loss 0.16764073073863983
epoch 1 step 400 loss 0.13667136430740356
epoch 1 step 600 loss 0.14606016874313354
epoch 1 step 800 loss 0.22875723242759705
epoch done
